In [1]:
# ==========================================================
# Notebook 3
# Consume TFRecords of Pass 2 + QUIC Classification Training
# ==========================================================

# CRITICAL: Set these BEFORE importing TensorFlow or JAX so they
# do not grab the TPU devices before TF gets a chance.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["JAX_PLATFORMS"] = "cpu"

import gc
import json
import random
import shutil
import warnings
import zipfile
from pathlib import Path
import time

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

# Install optuna early so all imports are available below
!pip -q install optuna

import optuna
from typing import Optional, Tuple, Dict, List
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.optimizers.schedules import LearningRateSchedule, CosineDecay
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("===== Notebook 3 =====")
print(f"TensorFlow : {tf.__version__}")
print(f"Numpy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("======================")

===== Notebook 3 =====
TensorFlow : 2.20.0
Numpy      : 2.0.2
Pandas     : 2.3.3


In [2]:
# ==========================================================
# Initialize hardware and dataset
# ==========================================================

print("Initializing...")

# ----------------------------------------------------------
# Distribution strategy  (TPU-first with evidence logging)
# ----------------------------------------------------------
tpu_name = os.environ.get("TPU_NAME", None)
DEVICE = "CPU"
strategy = tf.distribute.get_strategy()

try:
    if tpu_name:
        print(f"Found TPU_NAME={tpu_name}")
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu=tpu_name)
    else:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()

    print(f"TPU master: {resolver.master()}")
    print(f"TPU devices visible before init: {tf.config.list_logical_devices('TPU')}")

    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    DEVICE = "TPU"
    print("TPU system initialized successfully.")
except Exception as e:
    print(f"TPU initialization failed: {e}")
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        strategy = tf.distribute.MirroredStrategy()
        DEVICE = "GPU"
        print("Using GPU.")
    else:
        strategy = tf.distribute.get_strategy()
        DEVICE = "CPU"
        print("Using CPU.")

print(f"Device    : {DEVICE}")
print(f"Replicas  : {strategy.num_replicas_in_sync}")

# Mixed precision (set once, never again)
if DEVICE == "TPU":
    tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")
    print("Precision : mixed_bfloat16")
elif DEVICE == "GPU":
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Precision : mixed_float16")
else:
    tf.keras.mixed_precision.set_global_policy("float32")
    print("Precision : float32")

# ----------------------------------------------------------
# Adaptive config based on hardware (used downstream)
# ----------------------------------------------------------
DRY_RUN = False

if DEVICE == "TPU":
    BATCH_SIZE_PER_REPLICA = 64
    HPO_TRIALS = 0
    HPO_EPOCHS = 0
    FULL_EPOCHS = 15
    TOP_K_MODELS = 2
    TRAIN_STEPS_PER_EPOCH_CAP = 10_000

elif DEVICE == "GPU":
    BATCH_SIZE_PER_REPLICA = 256
    HPO_TRIALS = 0
    HPO_EPOCHS = 0
    FULL_EPOCHS = 15
    TOP_K_MODELS = 2
    TRAIN_STEPS_PER_EPOCH_CAP = 10_000

else:
    BATCH_SIZE_PER_REPLICA = 64
    HPO_TRIALS = 0
    HPO_EPOCHS = 0
    FULL_EPOCHS = 1
    TOP_K_MODELS = 1
    TRAIN_STEPS_PER_EPOCH_CAP = 100

    print("WARNING: CPU mode is verification-only.")

RUN_START_TIME = time.time()

# ~10.8 hours: safely below a typical 12-hour single-session limit,
# while no longer imposing the old ~2.6-hour stopper.
MAX_RUNTIME_SECONDS = 39_000
HPO_TIME_BUDGET_SECONDS = 0

# ==========================================================
# Dataset paths  (updated for the 30 M-row dataset)
# ==========================================================
DATASET_DIR = Path(
    "/kaggle/input/datasets/yahiaa1/cesnet-quicext25-tfrecord-full-0603/tfrecords_dataset/"
)

ARTIFACT_DIR = DATASET_DIR / "artficats"
METADATA_DIR = DATASET_DIR / "metadata"
TFRECORD_DIR = DATASET_DIR / "tfrecords"

# ==========================================================
# Load metadata
# ==========================================================
with open(METADATA_DIR / "dataset_metadata_202406_202503.json", "r") as f:
    metadata = json.load(f)

DATASET_NAME = metadata["dataset_name"]
DATASET_SIZE = metadata["dataset_size"]
TARGET_COLUMN = metadata["target_column"]
TRAIN_MONTHS = metadata["train_months"]
VAL_MONTH = metadata.get("val_month")
TEST_PERIOD = metadata["test_period"]
TRAINING_ROWS = metadata["training_rows"]
NUM_CLASSES = metadata["num_classes"]
NUM_AUX_FEATURES = metadata["num_aux_features"]
CLASS_NAMES = metadata.get("class_names")  # only present after the Pass 1 fix
PPI_SHAPE = tuple(metadata["ppi_shape"])
PPI_DTYPE = metadata["ppi_dtype"]
LABELS_ARE_CONTIGUOUS = metadata["labels_are_contiguous"]
# TRAIN_SHARDS = metadata["train_shards"]
VAL_SHARDS = metadata["validation_shards"]
TEST_SHARDS = metadata["test_shards"]

# ==========================================================
# Load preprocessing artifacts
# ==========================================================
preprocessing = joblib.load(ARTIFACT_DIR / "preprocessing_202406_202503.joblib")
iat_scaler = preprocessing["iat_scaler"]
size_scaler = preprocessing["size_scaler"]
aux_scaler = preprocessing["aux_scaler"]
aux_columns = preprocessing["aux_columns"]

class_weights = np.load(ARTIFACT_DIR / "class_weights_202406_202503.npy")

# ==========================================================
# TFRecord files
# ==========================================================
train_files = sorted(str(f) for f in TFRECORD_DIR.glob("train_*.tfrecord"))
val_files = sorted(str(f) for f in TFRECORD_DIR.glob("val_*.tfrecord"))
test_files = sorted(str(f) for f in TFRECORD_DIR.glob("test_*.tfrecord"))

# assert len(train_files) == TRAIN_SHARDS
assert len(val_files) == VAL_SHARDS
assert len(test_files) == TEST_SHARDS

# ==========================================================
# Summary
# ==========================================================
print()
print("Dataset loaded successfully.")
print(f"Dataset      : {DATASET_NAME}")
print(f"Size         : {DATASET_SIZE}")
print(f"Target       : {TARGET_COLUMN}")
print(f"Train rows   : {TRAINING_ROWS:,}")
print(f"Val month    : {VAL_MONTH}")
print(f"Classes      : {NUM_CLASSES}")
print(f"PPI shape    : {PPI_SHAPE}")
print(f"Aux features : {NUM_AUX_FEATURES}")
print()
print(f"Train shards : {len(train_files)}")
print(f"Val shards   : {len(val_files)}")
print(f"Test shards  : {len(test_files)}")
print()
print("Initialization completed.")

Initializing...
TPU initialization failed: Please provide a TPU Name to connect to.
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Using GPU.
Device    : GPU
Replicas  : 2
Precision : mixed_float16

Dataset loaded successfully.
Dataset      : CESNET-QUICEXT-25
Size         : M
Target       : APP
Train rows   : 27,727,267
Val month    : 202504
Classes      : 50
PPI shape    : (30, 3)
Aux features : 11

Train shards : 87
Val shards   : 8
Test shards  : 8

Initialization completed.


I0000 00:00:1791006582.350330      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791006582.358040      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
# ==========================================================
# tf.data input pipeline
# ==========================================================

print("Building tf.data pipeline...")

# ----------------------------------------------------------
# Training configuration
# ----------------------------------------------------------
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
AUTOTUNE = tf.data.AUTOTUNE
SHUFFLE_BUFFER = 100_000

# ----------------------------------------------------------
# TFRecord schema
# ----------------------------------------------------------
feature_description = {
    "ppi": tf.io.FixedLenFeature([], tf.string),
    "aux": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

# ----------------------------------------------------------
# Parse one example
# ----------------------------------------------------------
def parse_example(serialized):
    example = tf.io.parse_single_example(serialized, feature_description)

    ppi = tf.io.decode_raw(example["ppi"], tf.float32)
    ppi = tf.reshape(ppi, PPI_SHAPE)

    aux = tf.io.decode_raw(example["aux"], tf.float32)
    aux = tf.reshape(aux, (NUM_AUX_FEATURES,))

    label = tf.cast(example["label"], tf.int32)

    return {"ppi": ppi, "aux": aux}, label

# ----------------------------------------------------------
# Augmentation (training only)
# ----------------------------------------------------------
# def augment(features, label):
#     ppi = features["ppi"]
#     # Small jitter – robust to timing noise
#     ppi = ppi + tf.random.normal(tf.shape(ppi), stddev=0.008)
#     # Random per-channel scaling
#     scale = tf.random.uniform([1, 3], 0.97, 1.03)
#     ppi = ppi * scale
#     # Temporal shift (roll) – robust to packet reordering
#     shift = tf.random.uniform([], -3, 4, dtype=tf.int32)
#     ppi = tf.roll(ppi, shift, axis=0)
#     # Random time-step masking – simulates packet drops
#     mask = tf.random.uniform([PPI_SHAPE[0], 1]) > 0.02
#     ppi = ppi * tf.cast(mask, tf.float32)
#     return {"ppi": ppi, "aux": features["aux"]}, label
def augment_batch(features, label):
    ppi = features["ppi"]  # (B, 30, 3): IAT, DIR, SIZE
    real = tf.cast(tf.not_equal(ppi[:, :, 1:2], 0.0), tf.float32)

    iat_noise = tf.random.normal(tf.shape(ppi[:, :, 0:1]), stddev=0.008)
    size_noise = tf.random.normal(tf.shape(ppi[:, :, 2:3]), stddev=0.008)

    noise = tf.concat(
        [iat_noise, tf.zeros_like(ppi[:, :, 1:2]), size_noise], axis=2
    )

    iat_scale = tf.random.uniform(
        [tf.shape(ppi)[0], 1, 1], 0.97, 1.03
    )
    size_scale = tf.random.uniform(
        [tf.shape(ppi)[0], 1, 1], 0.97, 1.03
    )

    scale = tf.concat(
        [iat_scale, tf.ones_like(iat_scale), size_scale], axis=2
    )

    keep = tf.cast(
        tf.random.uniform(
            [tf.shape(ppi)[0], PPI_SHAPE[0], 1]
        ) > 0.02,
        tf.float32,
    )

    ppi = (ppi + noise) * scale * real * keep
    return {"ppi": ppi, "aux": features["aux"]}, label


def build_dataset(files, training=False):
    if training:
        files_ds = tf.data.Dataset.from_tensor_slices(files).shuffle(
            len(files), seed=SEED, reshuffle_each_iteration=True)
        ds = files_ds.interleave(
            lambda f: tf.data.TFRecordDataset(f),
            cycle_length=16, block_length=1,
            num_parallel_calls=AUTOTUNE, deterministic=False)
        ds = ds.map(parse_example, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(SHUFFLE_BUFFER, reshuffle_each_iteration=True).repeat()
        ds = ds.batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
        ds = ds.map(augment_batch, num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.TFRecordDataset(files, num_parallel_reads=AUTOTUNE)
        ds = ds.map(parse_example, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(GLOBAL_BATCH_SIZE)
        ds = ds.cache()
    return ds.prefetch(AUTOTUNE)


MONITOR_SAMPLES = 500_000
SELECTION_SAMPLES = 500_000
MONITOR_SHUFFLE_BUFFER = 2_600_000

def build_validation_sample(files, n_samples):
    ds = tf.data.TFRecordDataset(
        files,
        num_parallel_reads=AUTOTUNE
    )
    ds = ds.map(parse_example, num_parallel_calls=AUTOTUNE)
    ds = ds.shuffle(
        MONITOR_SHUFFLE_BUFFER,
        seed=SEED,
        reshuffle_each_iteration=False,
    )
    ds = ds.take(n_samples).cache()
    return ds.batch(GLOBAL_BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = build_dataset(train_files, training=True)
val_ds = build_dataset(val_files)
test_ds = build_dataset(test_files)

val_selection_ds = build_validation_sample(
    val_files,
    SELECTION_SAMPLES
)

# Use the same fixed 500k sample for epoch monitoring.
val_monitor_ds = val_selection_ds

print()
print("Datasets created.")
print(f"Global batch size    : {GLOBAL_BATCH_SIZE}")
print(f"Validation selection : {SELECTION_SAMPLES:,} flows")
print(f"Epoch monitor        : {MONITOR_SAMPLES:,} flows")
print()
print("tf.data pipeline ready.")

Building tf.data pipeline...

Datasets created.
Global batch size    : 512
Validation selection : 500,000 flows
Epoch monitor        : 500,000 flows

tf.data pipeline ready.


In [4]:
def scan(files):
    ds = (tf.data.TFRecordDataset(files, num_parallel_reads=AUTOTUNE)
          .map(parse_example, AUTOTUNE).batch(8192).prefetch(AUTOTUNE))
    nonfinite_ppi = nonfinite_aux = 0
    max_ppi = np.zeros(PPI_SHAPE[-1]); max_aux = np.zeros(NUM_AUX_FEATURES)
    for f, _ in ds:
        p, a = f["ppi"].numpy(), f["aux"].numpy()
        nonfinite_ppi += (~np.isfinite(p)).sum(); nonfinite_aux += (~np.isfinite(a)).sum()
        p = np.where(np.isfinite(p), p, 0); a = np.where(np.isfinite(a), a, 0)
        max_ppi = np.maximum(max_ppi, np.abs(p).max((0, 1))); max_aux = np.maximum(max_aux, np.abs(a).max(0))
    print("non-finite ppi/aux:", nonfinite_ppi, nonfinite_aux)
    print("max|ppi| per channel:", max_ppi); print("max|aux| per feature:", max_aux)

scan(train_files)
print("class_weights finite:", np.isfinite(class_weights).all(),
      "min/max:", class_weights.min(), class_weights.max())

non-finite ppi/aux: 0 0
max|ppi| per channel: [ 9.31669998  1.         12.51374531]
max|aux| per feature: [12.91474342  7.45841026  2.79438615  8.75649071  7.53182983 14.95975971
 10.97423363  8.12972546  5.37126446  4.26757145  4.58698463]
class_weights finite: True min/max: 0.25 5.0


In [5]:
# ==========================================================
# Verify tf.data pipeline
# ==========================================================

print("Verifying datasets...")

# ----------------------------------------------------------
# Read one batch
# ----------------------------------------------------------
features, labels = next(iter(train_ds))
ppi = features["ppi"]
aux = features["aux"]

print()
print("PPI")
print(f"  Shape : {ppi.shape}")
print(f"  DType : {ppi.dtype}")
print()
print("Auxiliary features")
print(f"  Shape : {aux.shape}")
print(f"  DType : {aux.dtype}")
print()
print("Labels")
print(f"  Shape : {labels.shape}")
print(f"  DType : {labels.dtype}")
print(f"  Min   : {int(tf.reduce_min(labels))}")
print(f"  Max   : {int(tf.reduce_max(labels))}")

# ----------------------------------------------------------
# Sanity checks
# ----------------------------------------------------------
assert tuple(ppi.shape[1:]) == PPI_SHAPE
assert tuple(aux.shape[1:]) == (NUM_AUX_FEATURES,)
assert ppi.dtype == tf.float32
assert aux.dtype == tf.float32
assert labels.dtype == tf.int32
assert int(tf.reduce_min(labels)) >= 0
assert int(tf.reduce_max(labels)) < NUM_CLASSES

print()
print("Pipeline verification completed.")

Verifying datasets...

PPI
  Shape : (512, 30, 3)
  DType : <dtype: 'float32'>

Auxiliary features
  Shape : (512, 11)
  DType : <dtype: 'float32'>

Labels
  Shape : (512,)
  DType : <dtype: 'int32'>
  Min   : 0
  Max   : 49

Pipeline verification completed.


In [6]:
# ==========================================================
# HPO + training utilities (shared across all architectures)
# ==========================================================

STEPS_PER_EPOCH = TRAINING_ROWS // GLOBAL_BATCH_SIZE
EPOCH_STEPS = TRAIN_STEPS_PER_EPOCH_CAP or STEPS_PER_EPOCH

def make_focal_loss(gamma: float, class_weights: np.ndarray, label_smoothing: float = 0.0):
    alpha = tf.constant(class_weights, dtype=tf.float32)

    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
        if label_smoothing > 0:
            oh = tf.one_hot(y_true, NUM_CLASSES)
            oh = oh * (1 - label_smoothing) + label_smoothing / NUM_CLASSES
            ce = -tf.reduce_sum(oh * tf.math.log(y_pred), axis=-1)
            p_t = tf.reduce_sum(oh * y_pred, axis=-1)
        else:
            idx = tf.stack([tf.range(tf.shape(y_true)[0]), y_true], axis=1)
            p_t = tf.gather_nd(y_pred, idx)
            ce = -tf.math.log(p_t)
        focal_weight = tf.pow(tf.clip_by_value(1.0 - p_t, 0.0, 1.0), gamma)
        loss = focal_weight * ce
        a_t = tf.gather(alpha, y_true)
        return tf.reduce_mean(a_t * loss)

    return loss_fn


class WarmupCosineDecay(LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps, alpha=0.05):
        super().__init__()
        self.peak_lr = peak_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.alpha = alpha
        self.cosine = CosineDecay(peak_lr, max(total_steps - warmup_steps, 1), alpha=alpha)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / tf.maximum(tf.cast(self.warmup_steps, tf.float32), 1.0))
        cosine_lr = self.cosine(tf.maximum(step - self.warmup_steps, 0.0))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "peak_lr": self.peak_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
            "alpha": self.alpha,
        }


class NaNGuard(Callback):
    """Abort on first NaN/Inf running loss, print diagnostics, and keep a
    'last finite' weight snapshot every `save_every` batches."""
    class Abort(Exception):
        pass

    def __init__(self, last_good_path=None, save_every=1000):
        super().__init__()
        self.last_good_path, self.save_every = last_good_path, save_every

    def _loss_scale(self):
        try:
            return float(self.model.optimizer.dynamic_scale.numpy())
        except Exception:
            return "n/a"

    def on_batch_end(self, batch, logs=None):
        loss = logs.get("loss") if logs else None
        if loss is not None and not np.isfinite(loss):
            bad_tr = [w.path for w in self.model.trainable_weights
                      if not np.isfinite(w.numpy()).all()]
            bad_nt = [w.path for w in self.model.non_trainable_weights
                      if not np.isfinite(w.numpy()).all()]
            print(f"\n[NaNGuard] batch {batch}: loss_scale={self._loss_scale()} | "
                  f"non-finite trainable={len(bad_tr)} {bad_tr[:3]} | "
                  f"non-finite non-trainable (BN stats)={len(bad_nt)} {bad_nt[:3]}")
            raise NaNGuard.Abort(f"NaN/Inf loss at batch {batch}")
        if self.last_good_path and (batch + 1) % self.save_every == 0:
            if all(np.isfinite(w).all() for w in self.model.get_weights()):
                self.model.save_weights(self.last_good_path)
class TimeBudget(Callback):
    def __init__(self, start_time, max_seconds):
        super().__init__(); self.start_time, self.max_seconds = start_time, max_seconds
    def on_epoch_end(self, epoch, logs=None):
        if time.time() - self.start_time > self.max_seconds:
            print(f"\nTime budget of {self.max_seconds/3600:.1f}h exceeded — stopping this model.")
            self.model.stop_training = True

def compute_macro_f1(model: Model, ds: tf.data.Dataset) -> float:
    """
    TPU-safe macro F1: uses model.predict(ds) (built-in distributed inference)
    instead of manually calling model(features) per batch in a Python loop.
    ds must be non-shuffled (val_ds/test_ds qualify) so the two independent
    passes below (predict, and manual label collection) stay aligned.
    """
    labels = np.concatenate([y.numpy() for _, y in ds])
    probs = model.predict(ds, verbose=0)
    preds = np.argmax(probs, axis=1)
    return f1_score(labels, preds, average="macro", zero_division=0)


class OptunaPruningF1Callback(Callback):
    """Reports truncated-val macro F1 to an Optuna trial each epoch; requests pruning if unpromising."""

    def __init__(self, trial: optuna.Trial, val_ds: tf.data.Dataset, max_val_batches: int = 150):
        super().__init__()
        self.trial = trial
        self.val_ds = val_ds.take(max_val_batches)
        self._labels = np.concatenate([y.numpy() for _, y in self.val_ds])
        self.best_f1 = -np.inf

    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.val_ds, verbose=0)
        preds = np.argmax(probs, axis=1)
        f1 = f1_score(self._labels, preds, average="macro", zero_division=0)
        self.best_f1 = max(self.best_f1, f1)
        logs = logs or {}
        logs["val_macro_f1_trunc"] = f1
        self.trial.report(f1, step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
def make_optimizer(cfg, lr_schedule):
    opt_cls = AdamW if cfg["optimizer"] == "adamw" else Adam
    kwargs = {"weight_decay": cfg["weight_decay"]} if cfg["optimizer"] == "adamw" else {}
    opt = opt_cls(learning_rate=lr_schedule, clipnorm=1.0, **kwargs)
    if DEVICE == "GPU":
        from tensorflow.keras.optimizers import LossScaleOptimizer
        # Modest fixed-ish scale: may halve on overflow, never grows back up.
        opt = LossScaleOptimizer(opt, initial_scale=2.0**12, dynamic_growth_steps=10**9)
    return opt

In [7]:
# ==========================================================
# Shared architectural blocks
# ==========================================================
def se_block(x, reduction=8, name="se"):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling1D(name=f"{name}_gap")(x)
    se = layers.Dense(max(filters // reduction, 4), activation="relu", name=f"{name}_fc1")(se)
    se = layers.Dense(filters, activation="sigmoid", dtype="float32", name=f"{name}_fc2")(se)
    se = layers.Reshape((1, filters), name=f"{name}_reshape")(se)
    return layers.Multiply(name=f"{name}_scale")([x, se])


def se_residual_block(x, filters, kernel_size, dropout_rate, activation, name):
    shortcut = x
    h = layers.Conv1D(filters, kernel_size, padding="same", name=f"{name}_conv1")(x)
    h = layers.BatchNormalization(name=f"{name}_bn1")(h)
    h = layers.Activation(activation, name=f"{name}_act1")(h)
    h = layers.SpatialDropout1D(dropout_rate, name=f"{name}_drop1")(h)
    h = layers.Conv1D(filters, kernel_size, padding="same", name=f"{name}_conv2")(h)
    h = layers.BatchNormalization(name=f"{name}_bn2")(h)
    h = se_block(h, name=f"{name}_se")
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same", name=f"{name}_proj")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_proj_bn")(shortcut)
    h = layers.Add(name=f"{name}_add")([shortcut, h])
    return layers.Activation(activation, name=f"{name}_act2")(h)


def transformer_encoder_block(x, num_heads, key_dim, ff_dim, dropout_rate, activation, name):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, name=f"{name}_mha")(x, x)
    attn = layers.Dropout(dropout_rate, name=f"{name}_mha_drop")(attn)
    x = layers.LayerNormalization(name=f"{name}_ln1")(layers.Add(name=f"{name}_add1")([x, attn]))
    ff = layers.Dense(ff_dim, activation=activation, name=f"{name}_ff1")(x)
    ff = layers.Dense(x.shape[-1], name=f"{name}_ff2")(ff)
    ff = layers.Dropout(dropout_rate, name=f"{name}_ff_drop")(ff)
    return layers.LayerNormalization(name=f"{name}_ln2")(layers.Add(name=f"{name}_add2")([x, ff]))


def add_aux_and_head(ppi_repr, aux_dim, dense_units, dropout_rate, activation, name_prefix):
    aux_input = Input(shape=(aux_dim,), name="aux")
    a = layers.Dense(64, activation=activation, name=f"{name_prefix}aux_dense1")(aux_input)
    a = layers.BatchNormalization(name=f"{name_prefix}aux_bn1")(a)
    a = layers.Dropout(0.2, name=f"{name_prefix}aux_drop1")(a)
    merged = layers.Concatenate(name=f"{name_prefix}merged")([ppi_repr, a])

    h = merged
    for i, units in enumerate(dense_units):
        h = layers.Dropout(dropout_rate, name=f"{name_prefix}head_drop{i}")(h)
        h = layers.Dense(units, activation=activation, name=f"{name_prefix}head_dense{i}")(h)
        h = layers.BatchNormalization(name=f"{name_prefix}head_bn{i}")(h)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="output")(h)
    return outputs, aux_input


class PositionalEmbedding(layers.Layer):
    """
    Trainable positional bias as a plain weight matrix, rather than an
    Embedding layer indexed by a bare tf.range(...) constant at graph-
    construction time. Structurally safer under strategy.scope()/TPU.
    """
    def __init__(self, seq_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len
        self.d_model = d_model

    def build(self, input_shape):
        self.pos_emb = self.add_weight(
            shape=(self.seq_len, self.d_model),
            initializer="random_normal",
            trainable=True,
            name="pos_emb",
        )

    def call(self, x):
        return x + self.pos_emb

    def get_config(self):
        config = super().get_config()
        config.update({"seq_len": self.seq_len, "d_model": self.d_model})
        return config


def build_model_a(cfg: dict) -> Model:
    ppi_input = Input(shape=PPI_SHAPE, name="ppi")
    x = ppi_input
    filters = cfg["base_filters"]
    for i in range(cfg["num_blocks"]):
        x = se_residual_block(x, filters, cfg["kernel_size"], cfg["dropout"], cfg["activation"], name=f"a_res{i}")
        if i < cfg["num_blocks"] - 1:
            x = layers.MaxPooling1D(2, name=f"a_pool{i}")(x)
        filters = min(filters * 2, 512)
    gap = layers.GlobalAveragePooling1D(name="a_gap")(x)
    gmp = layers.GlobalMaxPooling1D(name="a_gmp")(x)
    ppi_repr = layers.Concatenate(name="a_pooled")([gap, gmp])
    outputs, aux_input = add_aux_and_head(
        ppi_repr, NUM_AUX_FEATURES, cfg["dense_units"], cfg["dropout"], cfg["activation"], "a_"
    )
    return Model([ppi_input, aux_input], outputs, name="ModelA_SEResNet1D")


def build_model_b(cfg: dict) -> Model:
    ppi_input = Input(shape=PPI_SHAPE, name="ppi")
    x = ppi_input
    for i in range(cfg["num_blocks"]):
        dilation = 2 ** i
        h = layers.Conv1D(cfg["filters"], cfg["kernel_size"], padding="causal",
                           dilation_rate=dilation, name=f"b_conv{i}")(x)
        h = layers.BatchNormalization(name=f"b_bn{i}")(h)
        h = layers.Activation(cfg["activation"], name=f"b_act{i}")(h)
        h = layers.SpatialDropout1D(cfg["dropout"], name=f"b_drop{i}")(h)
        if x.shape[-1] != cfg["filters"]:
            x = layers.Conv1D(cfg["filters"], 1, padding="same", name=f"b_proj{i}")(x)
        x = layers.Add(name=f"b_add{i}")([x, h])
    gap = layers.GlobalAveragePooling1D(name="b_gap")(x)
    gmp = layers.GlobalMaxPooling1D(name="b_gmp")(x)
    ppi_repr = layers.Concatenate(name="b_pooled")([gap, gmp])
    outputs, aux_input = add_aux_and_head(
        ppi_repr, NUM_AUX_FEATURES, cfg["dense_units"], cfg["dropout"], cfg["activation"], "b_"
    )
    return Model([ppi_input, aux_input], outputs, name="ModelB_DilatedTCN")


def build_model_c(cfg: dict) -> Model:
    ppi_input = Input(shape=PPI_SHAPE, name="ppi")
    x = layers.Dense(cfg["d_model"], name="c_project")(ppi_input)
    x = PositionalEmbedding(PPI_SHAPE[0], cfg["d_model"], name="c_pos_emb")(x)
    for i in range(cfg["num_layers"]):
        x = transformer_encoder_block(
            x, cfg["num_heads"], cfg["d_model"] // cfg["num_heads"],
            cfg["d_model"] * cfg["ff_mult"], cfg["dropout"], cfg["activation"], name=f"c_tenc{i}"
        )
    gap = layers.GlobalAveragePooling1D(name="c_gap")(x)
    gmp = layers.GlobalMaxPooling1D(name="c_gmp")(x)
    ppi_repr = layers.Concatenate(name="c_pooled")([gap, gmp])
    outputs, aux_input = add_aux_and_head(
        ppi_repr, NUM_AUX_FEATURES, cfg["dense_units"], cfg["dropout"], cfg["activation"], "c_"
    )
    return Model([ppi_input, aux_input], outputs, name="ModelC_LightTransformer")


def build_model_d(cfg: dict) -> Model:
    ppi_input = Input(shape=PPI_SHAPE, name="ppi")
    x = ppi_input
    for i in range(cfg["num_conv_layers"]):
        x = layers.Conv1D(cfg["filters"], cfg["kernel_size"], padding="same", name=f"d_conv{i}")(x)
        x = layers.BatchNormalization(name=f"d_bn{i}")(x)
        x = layers.Activation(cfg["activation"], name=f"d_act{i}")(x)
        x = layers.SpatialDropout1D(cfg["dropout"], name=f"d_drop{i}")(x)

    attn_scores = layers.Dense(1, name="d_attn_scores")(x)
    attn_weights = layers.Softmax(axis=1, name="d_attn_softmax")(attn_scores)
    ppi_repr = layers.Multiply(name="d_attn_apply")([x, attn_weights])
    ppi_repr = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="d_attn_pool")(ppi_repr)

    outputs, aux_input = add_aux_and_head(
        ppi_repr, NUM_AUX_FEATURES, cfg["dense_units"], cfg["dropout"], cfg["activation"], "d_"
    )
    return Model([ppi_input, aux_input], outputs, name="ModelD_CNNAttentionPool")


MODEL_REGISTRY = {"A": build_model_a, "B": build_model_b, "C": build_model_c, "D": build_model_d}

In [8]:
# ==========================================================
# Optuna search spaces (one per architecture) + generic HPO driver
# ==========================================================
def sample_common(trial: optuna.Trial) -> dict:
    return {
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu", "swish"]),
        "dense_units": [trial.suggest_categorical("dense0", [128, 256, 384]),
                         trial.suggest_categorical("dense1", [64, 128, 192])],
        "peak_lr": trial.suggest_float("peak_lr", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamw"]),
        "focal_gamma": trial.suggest_float("focal_gamma", 0.5, 3.0),
        "label_smoothing": trial.suggest_float("label_smoothing", 0.0, 0.1),
    }

def sample_a(trial):
    cfg = sample_common(trial)
    cfg.update({
        "base_filters": trial.suggest_categorical("a_base_filters", [64, 96, 128]),
        "num_blocks": trial.suggest_int("a_num_blocks", 2, 3),
        "kernel_size": trial.suggest_categorical("a_kernel_size", [3, 5]),
    })
    return cfg

def sample_b(trial):
    cfg = sample_common(trial)
    cfg.update({
        "filters": trial.suggest_categorical("b_filters", [64, 96, 128]),
        "num_blocks": trial.suggest_int("b_num_blocks", 3, 5),
        "kernel_size": trial.suggest_categorical("b_kernel_size", [2, 3]),
    })
    return cfg

def sample_c(trial):
    cfg = sample_common(trial)
    cfg.update({
        "d_model": trial.suggest_categorical("c_d_model", [64, 96, 128]),
        "num_heads": trial.suggest_categorical("c_num_heads", [2, 4, 8]),
        "num_layers": trial.suggest_int("c_num_layers", 1, 3),
        "ff_mult": trial.suggest_categorical("c_ff_mult", [2, 4]),
    })
    return cfg

def sample_d(trial):
    cfg = sample_common(trial)
    cfg.update({
        "filters": trial.suggest_categorical("d_filters", [64, 96, 128]),
        "num_conv_layers": trial.suggest_int("d_num_conv_layers", 2, 4),
        "kernel_size": trial.suggest_categorical("d_kernel_size", [3, 5]),
    })
    return cfg

SAMPLERS = {"A": sample_a, "B": sample_b, "C": sample_c, "D": sample_d}

HPO_TRAIN_BATCHES = 20 if DRY_RUN else 800
HPO_VAL_BATCHES = 10 if DRY_RUN else 150

def run_hpo(model_key: str, n_trials: int = 20, epochs: int = 4) -> dict:
    build_fn = MODEL_REGISTRY[model_key]
    sample_fn = SAMPLERS[model_key]
    hpo_train_ds = train_ds.take(HPO_TRAIN_BATCHES)
    hpo_val_ds = val_monitor_ds.take(HPO_VAL_BATCHES)

    def objective(trial: optuna.Trial) -> float:
        cfg = sample_fn(trial)
        with strategy.scope():
            model = build_fn(cfg)
            lr_schedule = WarmupCosineDecay(cfg["peak_lr"], warmup_steps=100,
                                             total_steps=HPO_TRAIN_BATCHES * epochs)
            optimizer = make_optimizer(cfg, lr_schedule)
            model.compile(
                optimizer=optimizer,
                loss=make_focal_loss(cfg["focal_gamma"], class_weights, cfg["label_smoothing"]),
                metrics=["accuracy"],
            )
        pruning_cb = OptunaPruningF1Callback(trial, hpo_val_ds, max_val_batches=HPO_VAL_BATCHES)
        try:
            model.fit(hpo_train_ds, epochs=epochs, callbacks=[pruning_cb, NaNGuard()], verbose=0)
        except NaNGuard.Abort as e:
            print(f"  Trial hit NaN and was aborted: {e}")
        return pruning_cb.best_f1

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f"[{model_key}] best truncated val macro F1: {study.best_value:.4f}")
    return {"params": study.best_trial.params, "score": study.best_value}

In [9]:
!ls /kaggle/input/datasets/yahiaa1/best-params-checkpoint-v2

best_params_checkpoint.joblib


In [10]:
# ==========================================================
# Run HPO for all architectures
# ==========================================================
CKPT_PATH = Path("/kaggle/working/best_params_checkpoint.joblib")

PRIOR_CKPT = Path("/kaggle/input/datasets/yahiaa1/best-params-checkpoint-v2/best_params_checkpoint.joblib")  # <- adjust
if not CKPT_PATH.exists() and PRIOR_CKPT.exists():
    shutil.copy(PRIOR_CKPT, CKPT_PATH)

models_to_hpo = []

# Resume from checkpoint if TPU/GPU was preempted earlier
if CKPT_PATH.exists():
    hpo_results = joblib.load(CKPT_PATH)
    print(f"Resumed HPO checkpoint with models: {list(hpo_results.keys())}")
else:
    hpo_results = {}

for key in models_to_hpo:
    if time.time() - RUN_START_TIME > HPO_TIME_BUDGET_SECONDS:
        print(f"HPO time budget reached before Model {key} — stopping HPO early.")
        break
    if key in hpo_results:
        print(f"\n===== HPO: Model {key} (skipped, already in checkpoint) =====")
        continue
    print(f"\n===== HPO: Model {key} =====")
    hpo_results[key] = run_hpo(key, n_trials=HPO_TRIALS, epochs=HPO_EPOCHS)
    # Checkpoint to survive preemption / timeout
    joblib.dump(hpo_results, CKPT_PATH)
    print(f"Checkpoint saved after Model {key}")

for key, res in hpo_results.items():
    print(f"\nModel {key} best score: {res['score']:.4f}")
    for k, v in res["params"].items():
        print(f"  {k}: {v}")

# Freeze finalists from previously observed full-validation evidence.
models_to_train = [k for k in ("A", "C") if k in hpo_results]

best_params = {
    k: hpo_results[k]["params"]
    for k in models_to_train
}

if models_to_train != ["A", "C"]:
    raise RuntimeError(
        f"Expected frozen finalists A and C; found {models_to_train}"
    )

print(f"\n>>> Training frozen finalists: {models_to_train}")

Resumed HPO checkpoint with models: ['A', 'B', 'C', 'D']

Model A best score: 0.6469
  dropout: 0.21590058116550723
  activation: gelu
  dense0: 256
  dense1: 128
  peak_lr: 0.001558428978238547
  weight_decay: 0.00048783606034521424
  optimizer: adam
  focal_gamma: 1.0698379063548542
  label_smoothing: 0.042710778862625635
  a_base_filters: 96
  a_num_blocks: 3
  a_kernel_size: 3

Model B best score: 0.4277
  dropout: 0.21590058116550723
  activation: gelu
  dense0: 256
  dense1: 128
  peak_lr: 0.001558428978238547
  weight_decay: 0.00048783606034521424
  optimizer: adam
  focal_gamma: 1.0698379063548542
  label_smoothing: 0.042710778862625635
  b_filters: 96
  b_num_blocks: 4
  b_kernel_size: 2

Model C best score: 0.4580
  dropout: 0.16974657160199658
  activation: swish
  dense0: 256
  dense1: 64
  peak_lr: 0.0009437923703573488
  weight_decay: 0.0002829219225536188
  optimizer: adam
  focal_gamma: 1.1046307272511293
  label_smoothing: 0.009310276780589922
  c_d_model: 96
  c_num_h

In [11]:
# ==========================================================
# Reconstruct full config dicts from Optuna's best_params (flat dict ->
# nested cfg expected by build_model_*), then train each selected model fully.
# ==========================================================
def params_to_cfg(model_key: str, params: dict) -> dict:
    cfg = {
        "dropout": params["dropout"],
        "activation": params["activation"],
        "dense_units": [params["dense0"], params["dense1"]],
        "peak_lr": params["peak_lr"],
        "weight_decay": params["weight_decay"],
        "optimizer": params["optimizer"],
        "focal_gamma": params["focal_gamma"],
        "label_smoothing": params["label_smoothing"],
    }
    if model_key == "A":
        cfg.update(base_filters=params["a_base_filters"], num_blocks=params["a_num_blocks"],
                    kernel_size=params["a_kernel_size"])
    elif model_key == "B":
        cfg.update(filters=params["b_filters"], num_blocks=params["b_num_blocks"],
                    kernel_size=params["b_kernel_size"])
    elif model_key == "C":
        cfg.update(d_model=params["c_d_model"], num_heads=params["c_num_heads"],
                    num_layers=params["c_num_layers"], ff_mult=params["c_ff_mult"])
    elif model_key == "D":
        cfg.update(filters=params["d_filters"], num_conv_layers=params["d_num_conv_layers"],
                    kernel_size=params["d_kernel_size"])
    return cfg


class ValMacroF1(Callback):
    """Macro-F1 on the monitor subset each epoch; labels are read once."""
    def __init__(self, ds):
        super().__init__()
        self.ds = ds
        self.labels = np.concatenate([y.numpy() for _, y in ds])

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        preds = np.argmax(self.model.predict(self.ds, verbose=0), axis=1)
        f1 = f1_score(self.labels, preds, average="macro", zero_division=0)
        logs["val_macro_f1"] = f1
        print(f"  val_macro_f1: {f1:.4f}  ({time.time() - RUN_START_TIME:,.0f}s elapsed)")

CHECKPOINT_DIR = Path("/kaggle/working/model_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
MAX_NAN_RETRIES = 3

class EpochTracker(Callback):
    def __init__(self):
        super().__init__(); self.done = 0
    def on_epoch_end(self, epoch, logs=None):
        self.done = epoch + 1
        
def train_full_model(model_key: str):
    cfg = params_to_cfg(model_key, best_params[model_key])
    with strategy.scope():
        model = MODEL_REGISTRY[model_key](cfg)
        lr_schedule = WarmupCosineDecay(cfg["peak_lr"], warmup_steps=EPOCH_STEPS * 2,
                                         total_steps=EPOCH_STEPS * FULL_EPOCHS)
        optimizer = make_optimizer(cfg, lr_schedule)
        model.compile(optimizer=optimizer,
                      loss=make_focal_loss(cfg["focal_gamma"], class_weights, cfg["label_smoothing"]),
                      metrics=["accuracy"])

    print(f"\n===== Training {model.name} (full) =====")
    early_stop = EarlyStopping(monitor="val_macro_f1", patience=8, mode="max",
                               restore_best_weights=True, verbose=1)
    ckpt_path = CHECKPOINT_DIR / f"{model.name}_best_on_disk.weights.h5"
    ckpt = ModelCheckpoint(str(ckpt_path), save_weights_only=True,
                           monitor="val_macro_f1", mode="max", save_best_only=True)
    last_good = CHECKPOINT_DIR / f"{model.name}_last_good.weights.h5"
    last_good.unlink(missing_ok=True)
    tracker = EpochTracker()
    for attempt in range(MAX_NAN_RETRIES + 1):
        try:
            model.fit(train_ds, epochs=FULL_EPOCHS, initial_epoch=tracker.done,
                      steps_per_epoch=EPOCH_STEPS,
                      callbacks=[ValMacroF1(val_monitor_ds), tracker, early_stop,
                                 NaNGuard(last_good),
                                 TimeBudget(RUN_START_TIME, MAX_RUNTIME_SECONDS), ckpt],
                      verbose=1)
            break
        except NaNGuard.Abort as e:
            print(f"{model.name}: {e} (attempt {attempt + 1}/{MAX_NAN_RETRIES + 1})")
            if not last_good.exists():
                print(f"{model.name}: no finite snapshot yet -- giving up on this model.")
                break
            model.load_weights(last_good)
            print(f"{model.name}: restored last finite weights, resuming with fresh data order.")
            
    if ckpt_path.exists():
        model.load_weights(ckpt_path)
        if any(np.isnan(w).any() for w in model.get_weights()):
            print(f"WARNING: {model.name}: checkpoint is unexpectedly NaN -- discarding this model.")
            return None
        model.save_weights(CHECKPOINT_DIR / f"{model.name}_best.weights.h5")
        return model
    print(f"WARNING: {model.name}: no epoch completed cleanly -- discarding this model.")
    return None


trained_models = {}
for key in models_to_train:
    if time.time() - RUN_START_TIME > MAX_RUNTIME_SECONDS:
        print(f"Time budget exhausted before Model {key} — stopping with {len(trained_models)} model(s) done.")
        break
    result = train_full_model(key)
    if result is not None:
        trained_models[key] = result
    gc.collect()
assert len(trained_models) > 0, "No model completed training successfully."


===== Training ModelA_SEResNet1D (full) =====
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica

In [12]:
# ==========================================================
# Collect validation probabilities
# Test data remains untouched until the ensemble method is frozen.
# ==========================================================

def collect_probs(model, ds):
    labels = np.concatenate(
        [y.numpy() for _, y in ds]
    )
    probs = model.predict(ds, verbose=0)
    return probs, labels

val_ds_eval = val_selection_ds
val_probs = {}
y_val_full = None

for key, model in trained_models.items():
    print(f"Collecting validation probabilities for Model {key}...")
    val_probs[key], y_val_full = collect_probs(
        model, val_ds_eval
    )

bad_keys = [
    k for k in val_probs
    if np.isnan(val_probs[k]).any()
]

for k in bad_keys:
    print(
        f"WARNING: Model {k} has NaN validation "
        "predictions -- excluding it."
    )
    del val_probs[k]
    del trained_models[k]

assert len(trained_models) > 0

for key, p in val_probs.items():
    score = f1_score(
        y_val_full,
        np.argmax(p, axis=1),
        average="macro",
        zero_division=0,
    )
    print(
        f"Model {key}: val_probs {p.shape}, "
        f"individual macro F1 = {score:.4f}"
    )

Model A: val_probs (500000, 50), individual macro F1 = 0.8395
Model C: val_probs (500000, 50), individual macro F1 = 0.8210


In [13]:
# ==========================================================
# CELL 12 — Ensemble selection on April 2025 validation
# ==========================================================
#
# Protocol:
#   1. Draw a fixed random 500k-sample subset from April.
#   2. Split that subset into stratified fit/holdout halves.
#   3. Compare:
#        - single A
#        - single C
#        - uniform A+C
#        - weighted A+C
#        - logistic-regression stacking A+C
#   4. Select the best method using ONLY the holdout half.
#   5. Refit the selected ensemble rule on the complete 500k
#      validation-selection sample.
#
# IMPORTANT:
#   No test data is accessed anywhere in this cell.
# ==========================================================

# ==========================================================
# Validation-only ensemble selection
# ==========================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

trained_keys = list(trained_models.keys())
assert trained_keys == ["A", "C"]

X_meta_val = np.hstack(
    [val_probs[k] for k in trained_keys]
).astype(np.float32, copy=False)

idx = np.arange(len(y_val_full))

idx_a, idx_b = train_test_split(
    idx,
    test_size=0.5,
    random_state=SEED,
    stratify=y_val_full,
)

# ----- Single-model references -----
single_scores = {}

for key in trained_keys:
    single_scores[f"single_{key}"] = f1_score(
        y_val_full[idx_b],
        np.argmax(val_probs[key][idx_b], axis=1),
        average="macro",
        zero_division=0,
    )

# ----- Uniform averaging -----
uniform_prob = np.mean(
    [val_probs[k][idx_b] for k in trained_keys],
    axis=0,
)

uniform_val_f1 = f1_score(
    y_val_full[idx_b],
    np.argmax(uniform_prob, axis=1),
    average="macro",
    zero_division=0,
)

# ----- Weighted averaging -----
best_weight = 1.0
best_weight_f1 = -np.inf

for w_a in np.linspace(0.0, 1.0, 101):
    ens = (
        w_a * val_probs["A"][idx_a]
        + (1.0 - w_a) * val_probs["C"][idx_a]
    )

    score = f1_score(
        y_val_full[idx_a],
        np.argmax(ens, axis=1),
        average="macro",
        zero_division=0,
    )

    if score > best_weight_f1:
        best_weight_f1 = score
        best_weight = float(w_a)

weighted_prob_b = (
    best_weight * val_probs["A"][idx_b]
    + (1.0 - best_weight) * val_probs["C"][idx_b]
)

weighted_val_f1 = f1_score(
    y_val_full[idx_b],
    np.argmax(weighted_prob_b, axis=1),
    average="macro",
    zero_division=0,
)

# ----- Logistic-regression stacking -----
meta_model = LogisticRegression(
    max_iter=300,
    C=0.5,
    solver="lbfgs",
)

meta_model.fit(
    X_meta_val[idx_a],
    y_val_full[idx_a],
)

stack_val_f1 = f1_score(
    y_val_full[idx_b],
    meta_model.predict(X_meta_val[idx_b]),
    average="macro",
    zero_division=0,
)

candidate_scores = {
    **single_scores,
    "uniform_A_C": uniform_val_f1,
    "weighted_A_C": weighted_val_f1,
    "stacking_A_C": stack_val_f1,
}

selected_method = max(
    candidate_scores,
    key=candidate_scores.get,
)

print("\nValidation-selection scores:")
for name, score in candidate_scores.items():
    print(f"  {name:16s}: {score:.4f}")

print(
    f"Best training-half weighted coefficient for A: "
    f"{best_weight:.2f}"
)

print(f"Selected method: {selected_method}")

# ----- Refit selected rule on ALL 500k validation samples -----

final_meta_model = None
final_weights = None

if selected_method == "stacking_A_C":
    final_meta_model = LogisticRegression(
        max_iter=300,
        C=0.5,
        solver="lbfgs",
    )

    final_meta_model.fit(
        X_meta_val,
        y_val_full,
    )

elif selected_method == "weighted_A_C":

    best_weight_all = 1.0
    best_weight_all_f1 = -np.inf

    for w_a in np.linspace(0.0, 1.0, 101):
        ens = (
            w_a * val_probs["A"]
            + (1.0 - w_a) * val_probs["C"]
        )

        score = f1_score(
            y_val_full,
            np.argmax(ens, axis=1),
            average="macro",
            zero_division=0,
        )

        if score > best_weight_all_f1:
            best_weight_all_f1 = score
            best_weight_all = float(w_a)

    final_weights = np.array(
        [best_weight_all, 1.0 - best_weight_all],
        dtype=np.float32,
    )

elif selected_method == "uniform_A_C":
    final_weights = np.array(
        [0.5, 0.5],
        dtype=np.float32,
    )

print("Selected ensemble rule has been frozen.")
# ----------------------------------------------------------
# Normalize final ensemble metadata for downstream cells
# ----------------------------------------------------------

if selected_method == "weighted_A_C":
    final_weight_A = float(final_weights[0])
    final_weight_C = float(final_weights[1])

elif selected_method == "uniform_A_C":
    final_weight_A = 0.5
    final_weight_C = 0.5

elif selected_method == "single_A":
    final_weight_A = 1.0
    final_weight_C = 0.0

elif selected_method == "single_C":
    final_weight_A = 0.0
    final_weight_C = 1.0

else:
    # Stacking does not use scalar model weights.
    final_weight_A = None
    final_weight_C = None

final_single_key = (
    selected_method
    if selected_method in {"single_A", "single_C"}
    else None
)


Validation-selection scores:
  single_A        : 0.8404
  single_C        : 0.8243
  uniform_A_C     : 0.8486
  weighted_A_C    : 0.8489
  stacking_A_C    : 0.8574
Best training-half weighted coefficient for A: 0.52
Selected method: stacking_A_C
Selected ensemble rule has been frozen.


In [14]:
# ==========================================================
# CELL 13 — FINAL TEST EVALUATION
# ==========================================================
#
# IMPORTANT:
#   This cell is the FIRST cell that accesses test_ds.
#
#   The ensemble method was completely frozen in Cell 12.
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ----------------------------------------------------------
# Safety checks
# ----------------------------------------------------------
if selected_method not in {
    "single_A",
    "single_C",
    "uniform_A_C",
    "weighted_A_C",
    "stacking_A_C",
}:
    raise RuntimeError(
        f"Invalid selected method: {selected_method}"
    )

if not trained_models:
    raise RuntimeError(
        "No trained models are available."
    )

# ----------------------------------------------------------
# Apply the frozen ensemble without creating enormous
# test-time meta arrays.
# ----------------------------------------------------------
def combine_probabilities(
    prob_A,
    prob_C,
    method,
    batch_size=100_000,
):
    """
    Applies the already-frozen ensemble rule.

    This function processes rows in chunks so that stacking
    never creates an enormous full test meta-matrix.
    """

    n = len(prob_A)
    output_parts = []

    for start in range(
        0,
        n,
        batch_size,
    ):

        stop = min(
            start + batch_size,
            n,
        )

        a = prob_A[start:stop]
        c = prob_C[start:stop]

        if method == "stacking_A_C":

            meta_x = np.concatenate(
                [a, c],
                axis=1,
            ).astype(
                np.float32,
                copy=False,
            )

            part = final_meta_model.predict_proba(
                meta_x
            )

        elif method == "weighted_A_C":

            part = (
                final_weight_A * a
                + final_weight_C * c
            )

        elif method == "uniform_A_C":

            part = (
                0.5 * a
                + 0.5 * c
            )

        elif method == "single_A":

            part = a

        elif method == "single_C":

            part = c

        else:
            raise RuntimeError(
                f"Unknown ensemble method: {method}"
            )

        output_parts.append(
            np.asarray(
                part,
                dtype=np.float32,
            )
        )

    return np.concatenate(
        output_parts,
        axis=0,
    )


# ==========================================================
# Read May 2025 test predictions
# ==========================================================

test_probs = {}
y_test_full = None

for key in ("A", "C"):

    print(
        f"Collecting FINAL test probabilities "
        f"for Model {key}..."
    )

    probs_this_model, labels_this_model = (
        collect_probs(
            trained_models[key],
            test_ds,
        )
    )

    test_probs[key] = (
        probs_this_model.astype(
            np.float32,
            copy=False,
        )
    )

    if y_test_full is None:

        y_test_full = labels_this_model

    elif not np.array_equal(
        y_test_full,
        labels_this_model,
    ):

        raise RuntimeError(
            "Test labels are not identical between "
            f"Model A and Model {key}."
        )

print()
print(
    f"Test A probabilities : {test_probs['A'].shape}"
)
print(
    f"Test C probabilities : {test_probs['C'].shape}"
)
print(
    f"Test flows           : {len(y_test_full):,}"
)

# ==========================================================
# Final prediction
# ==========================================================

final_test_probs = combine_probabilities(
    test_probs["A"],
    test_probs["C"],
    selected_method,
)

final_test_predictions = np.argmax(
    final_test_probs,
    axis=1,
)

# ==========================================================
# Metrics
# ==========================================================

accuracy = accuracy_score(
    y_test_full,
    final_test_predictions,
)

precision_macro = precision_score(
    y_test_full,
    final_test_predictions,
    average="macro",
    zero_division=0,
)

recall_macro = recall_score(
    y_test_full,
    final_test_predictions,
    average="macro",
    zero_division=0,
)

f1_macro = f1_score(
    y_test_full,
    final_test_predictions,
    average="macro",
    zero_division=0,
)

f1_weighted = f1_score(
    y_test_full,
    final_test_predictions,
    average="weighted",
    zero_division=0,
)

# ==========================================================
# Expected Calibration Error
# ==========================================================

def expected_calibration_error(
    y_true,
    probs,
    n_bins=10,
):
    confidence = np.max(
        probs,
        axis=1,
    )

    predictions = np.argmax(
        probs,
        axis=1,
    )

    correctness = (
        predictions == y_true
    ).astype(
        np.float64
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for i in range(n_bins):

        left = bin_edges[i]
        right = bin_edges[i + 1]

        if i == 0:

            mask = (
                (confidence >= left)
                & (confidence <= right)
            )

        else:

            mask = (
                (confidence > left)
                & (confidence <= right)
            )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        bin_accuracy = (
            correctness[mask].mean()
        )

        bin_confidence = (
            confidence[mask].mean()
        )

        ece += (
            count / len(y_true)
        ) * abs(
            bin_accuracy
            - bin_confidence
        )

    return float(ece)


ece_10bin = expected_calibration_error(
    y_test_full,
    final_test_probs,
    n_bins=10,
)

# ==========================================================
# Print aggregate results
# ==========================================================

print()
print("==========================================================")
print("FINAL TEST RESULTS")
print("==========================================================")
print(
    f"Selected method       : {selected_method}"
)
print(
    f"Accuracy              : {accuracy:.4f}"
)
print(
    f"Precision (macro)     : {precision_macro:.4f}"
)
print(
    f"Recall (macro)        : {recall_macro:.4f}"
)
print(
    f"F1 (macro)            : {f1_macro:.4f}"
)
print(
    f"F1 (weighted)         : {f1_weighted:.4f}"
)
print(
    f"ECE (10-bin)          : {ece_10bin:.4f}"
)
print(
    f"Test flows            : "
    f"{len(y_test_full):,}"
)

if selected_method == "weighted_A_C":

    print(
        f"Final A weight        : "
        f"{final_weight_A:.2f}"
    )

    print(
        f"Final C weight        : "
        f"{final_weight_C:.2f}"
    )

# ==========================================================
# Per-class results
# ==========================================================

target_names = (
    CLASS_NAMES
    if CLASS_NAMES is not None
    else [
        str(i)
        for i in range(NUM_CLASSES)
    ]
)

print()
print("==========================================================")
print("PER-CLASS TEST REPORT")
print("==========================================================")

print(
    classification_report(
        y_test_full,
        final_test_predictions,
        labels=np.arange(NUM_CLASSES),
        target_names=target_names,
        digits=4,
        zero_division=0,
    )
)

# ==========================================================
# Confusion matrix
# ==========================================================

conf_mat = confusion_matrix(
    y_test_full,
    final_test_predictions,
    labels=np.arange(NUM_CLASSES),
)

np.save(
    "/kaggle/working/final_confusion_matrix.npy",
    conf_mat,
)

np.save(
    "/kaggle/working/final_test_predictions.npy",
    final_test_predictions,
)

# Optional: save probabilities for later analysis.
np.save(
    "/kaggle/working/final_test_probabilities.npy",
    final_test_probs,
)

# ==========================================================
# Save machine-readable final metrics
# ==========================================================

final_metrics = {
    "selected_method": selected_method,
    "accuracy": float(accuracy),
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "ece_10bin": float(ece_10bin),
    "test_flows": int(len(y_test_full)),
    "final_weight_A": (
        None
        if final_weight_A is None
        else float(final_weight_A)
    ),
    "final_weight_C": (
        None
        if final_weight_C is None
        else float(final_weight_C)
    ),
    "single_key": final_single_key,
}

with open(
    "/kaggle/working/final_test_metrics.json",
    "w",
) as f:
    json.dump(
        final_metrics,
        f,
        indent=2,
    )

print()
print(
    "Final test evaluation complete."
)
print(
    "Metrics, predictions, probabilities, and "
    "confusion matrix saved in /kaggle/working."
)


Test A probabilities : (2359061, 50)
Test C probabilities : (2359061, 50)
Test flows           : 2,359,061

FINAL TEST RESULTS
Selected method       : stacking_A_C
Accuracy              : 0.9324
Precision (macro)     : 0.8725
Recall (macro)        : 0.8272
F1 (macro)            : 0.8386
F1 (weighted)         : 0.9320
ECE (10-bin)          : 0.0096
Test flows            : 2,359,061

PER-CLASS TEST REPORT
                       precision    recall  f1-score   support

   apple-privaterelay     0.9999    0.9999    0.9999     87993
 chrome-remotedesktop     0.9908    0.8398    0.9091       256
     cloudflare-cdnjs     0.7606    0.7629    0.7618       658
            connectad     0.9449    0.9109    0.9276      1223
              discord     0.9915    0.9937    0.9926     29738
                dm-de     0.7405    0.9073    0.8155       151
              dns-doh     0.9905    0.9425    0.9659     26975
     facebook-connect     0.9352    0.8630    0.8976      5802
       facebook-graph   

In [15]:
# ==========================================================
# CELL 14 — EXPORT MODELS + ENSEMBLE METADATA
# ==========================================================

EXPORT_DIR = Path(
    "/kaggle/working/exported_models"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ----------------------------------------------------------
# Export base models
# ----------------------------------------------------------

for key in ("A", "C"):

    model = trained_models[key]

    export_path = (
        EXPORT_DIR
        / f"model_{key}_savedmodel"
    )

    # Make reruns safe.
    if export_path.exists():

        shutil.rmtree(
            export_path,
            ignore_errors=True,
        )

    print(
        f"Exporting Model {key} -> "
        f"{export_path}"
    )

    model.export(
        str(export_path)
    )

# ----------------------------------------------------------
# Export stacking model if stacking was selected
# ----------------------------------------------------------

if (
    selected_method == "stacking_A_C"
    and final_meta_model is not None
):

    stacking_path = (
        EXPORT_DIR
        / "stacking_logistic_regression.joblib"
    )

    joblib.dump(
        final_meta_model,
        stacking_path,
    )

    print(
        f"Saved stacking model -> "
        f"{stacking_path}"
    )

# ----------------------------------------------------------
# Save the complete ensemble configuration
# ----------------------------------------------------------

export_metadata = {
    "base_models": [
        "A",
        "C",
    ],

    "selected_method": selected_method,

    "weight_A": (
        None
        if final_weight_A is None
        else float(final_weight_A)
    ),

    "weight_C": (
        None
        if final_weight_C is None
        else float(final_weight_C)
    ),

    "single_model": final_single_key,

    "num_classes": int(
        NUM_CLASSES
    ),

    "class_names": (
        CLASS_NAMES
        if CLASS_NAMES is not None
        else [
            str(i)
            for i in range(NUM_CLASSES)
        ]
    ),

    "ppi_shape": [
        int(x)
        for x in PPI_SHAPE
    ],

    "num_aux_features": int(
        NUM_AUX_FEATURES
    ),

    "batch_size": int(
        GLOBAL_BATCH_SIZE
    ),

    "tta": False,

    "ensemble_selection": {
        "validation_month": "2025-04",
        "selection_protocol": (
            "fixed stratified 500k sample; "
            "50/50 fit-holdout"
        ),
    },

    "test_protocol": (
        "May 2025 test accessed only after "
        "ensemble method was frozen"
    ),

    "metrics_file": (
        "/kaggle/working/final_test_metrics.json"
    ),

    "confusion_matrix_file": (
        "/kaggle/working/final_confusion_matrix.npy"
    ),
}

metadata_path = (
    EXPORT_DIR
    / "ensemble_metadata.json"
)

with open(
    metadata_path,
    "w",
) as f:

    json.dump(
        export_metadata,
        f,
        indent=2,
    )

print()
print("==========================================================")
print("EXPORT COMPLETE")
print("==========================================================")

for path in sorted(
    EXPORT_DIR.iterdir()
):

    print(
        f"  {path}"
    )

Exporting Model A -> /kaggle/working/exported_models/model_A_savedmodel
INFO:tensorflow:Assets written to: /kaggle/working/exported_models/model_A_savedmodel/assets


INFO:tensorflow:Assets written to: /kaggle/working/exported_models/model_A_savedmodel/assets


Saved artifact at '/kaggle/working/exported_models/model_A_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 30, 3), dtype=tf.float32, name='ppi'), TensorSpec(shape=(None, 11), dtype=tf.float32, name='aux')]
Output Type:
  TensorSpec(shape=(None, 50), dtype=tf.float32, name=None)
Captures:
  137302331840208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695559248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695560400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695560592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695557136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695560016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695560784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695556176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301094122896: TensorSpec(shape=(), dtype=tf.resource, name=None)


INFO:tensorflow:Assets written to: /kaggle/working/exported_models/model_C_savedmodel/assets


Saved artifact at '/kaggle/working/exported_models/model_C_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 30, 3), dtype=tf.float32, name='ppi'), TensorSpec(shape=(None, 11), dtype=tf.float32, name='aux')]
Output Type:
  TensorSpec(shape=(None, 50), dtype=tf.float32, name=None)
Captures:
  137301695556944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301695558096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301092560016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301090810768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301090808656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301090810000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301090811344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301090810960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137301090808848: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [16]:
# ==========================================================
# CELL 15 — FINAL ENSEMBLE INFERENCE BENCHMARK
# ==========================================================
#
# Unlike the old Cell 14, this measures the ACTUAL selected
# deployment path:
#
#   Model A
#       +
#   Model C
#       +
#   selected ensemble operation
#
# Therefore this is NOT merely a single-model upper bound.
# ==========================================================

# ----------------------------------------------------------
# Get one real batch from the test pipeline.
# ----------------------------------------------------------

sample_features = next(
    iter(
        test_ds.take(1)
    )
)[0]

sample_batch = sample_features

# ----------------------------------------------------------
# Final ensemble prediction for one batch
# ----------------------------------------------------------

def predict_final_batch(batch):
    """
    Execute the complete frozen inference pipeline
    for one batch.
    """

    # Model A
    pA = tf.cast(
        trained_models["A"](
            batch,
            training=False,
        ),
        tf.float32,
    ).numpy()

    # Model C
    pC = tf.cast(
        trained_models["C"](
            batch,
            training=False,
        ),
        tf.float32,
    ).numpy()

    # ------------------------------------------------------
    # Frozen ensemble rule
    # ------------------------------------------------------

    if selected_method == "stacking_A_C":

        meta_x = np.concatenate(
            [
                pA,
                pC,
            ],
            axis=1,
        ).astype(
            np.float32,
            copy=False,
        )

        result = (
            final_meta_model.predict_proba(
                meta_x
            )
        )

    elif selected_method == "weighted_A_C":

        result = (
            final_weight_A * pA
            + final_weight_C * pC
        )

    elif selected_method == "uniform_A_C":

        result = (
            0.5 * pA
            + 0.5 * pC
        )

    elif selected_method == "single_A":

        result = pA

    elif selected_method == "single_C":

        result = pC

    else:

        raise RuntimeError(
            f"Unknown selected method: "
            f"{selected_method}"
        )

    return result


# ==========================================================
# Warm-up
# ==========================================================

print(
    "Warming up final inference pipeline..."
)

for _ in range(5):

    _ = predict_final_batch(
        sample_batch
    )


# ==========================================================
# Timed benchmark
# ==========================================================

N_BENCHMARK_BATCHES = 50

start = time.perf_counter()

for _ in range(
    N_BENCHMARK_BATCHES
):

    _ = predict_final_batch(
        sample_batch
    )

elapsed = (
    time.perf_counter()
    - start
)

# ----------------------------------------------------------
# Metrics
# ----------------------------------------------------------

batch_size = int(
    sample_batch["ppi"].shape[0]
)

latency_ms = (
    elapsed
    / N_BENCHMARK_BATCHES
    * 1000.0
)

throughput = (
    batch_size
    * N_BENCHMARK_BATCHES
    / elapsed
)

# ----------------------------------------------------------
# Print
# ----------------------------------------------------------

print()
print("==========================================================")
print("FINAL ENSEMBLE INFERENCE BENCHMARK")
print("==========================================================")

print(
    f"Selected method       : "
    f"{selected_method}"
)

print(
    f"Batch size            : "
    f"{batch_size}"
)

print(
    f"Mean latency          : "
    f"{latency_ms:.2f} ms/batch"
)

print(
    f"Throughput            : "
    f"{throughput:,.0f} flows/sec"
)

print(
    f"Benchmark batches     : "
    f"{N_BENCHMARK_BATCHES}"
)

print()
print(
    "This benchmark includes Model A + Model C "
    "and the complete selected ensemble operation."
)

# ==========================================================
# Save benchmark results
# ==========================================================

benchmark_results = {
    "selected_method": selected_method,
    "batch_size": int(batch_size),
    "latency_ms_per_batch": float(
        latency_ms
    ),
    "throughput_flows_per_sec": float(
        throughput
    ),
    "benchmark_batches": int(
        N_BENCHMARK_BATCHES
    ),
}

with open(
    "/kaggle/working/final_inference_benchmark.json",
    "w",
) as f:

    json.dump(
        benchmark_results,
        f,
        indent=2,
    )

print(
    "Benchmark saved to "
    "/kaggle/working/final_inference_benchmark.json"
)

Warming up final inference pipeline...

FINAL ENSEMBLE INFERENCE BENCHMARK
Selected method       : stacking_A_C
Batch size            : 512
Mean latency          : 174.26 ms/batch
Throughput            : 2,938 flows/sec
Benchmark batches     : 50

This benchmark includes Model A + Model C and the complete selected ensemble operation.
Benchmark saved to /kaggle/working/final_inference_benchmark.json
